In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import models, transforms
from PIL import Image
import io
import os
import base64
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [2]:
STYLE_PATHS = {
    "Seurat - La Grande Jatte": "/kaggle/input/styleimage/ASundayonLaGrandeJatte_GeorgesSeurat.jpg",
    "Munch - The Scream": "/kaggle/input/styleimage/EdvardMunch_TheScream.jpg",
    "Picasso - Weeping Woman": "/kaggle/input/styleimage/Picasso_TheWeepingWoman.jpg",
    "Hokusai - Tsunami": "/kaggle/input/styleimage/Tsunami_hokusai.jpg",
    "Van Gogh - Starry Night": "/kaggle/input/styleimage/VanGogh_StarryNight.jpg"
}

In [3]:
# Determine the available device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load VGG19 with modern 2025 weights argument
# pretrained=True is legacy; weights='DEFAULT' is recommended
pretrained_net = torchvision.models.vgg19(weights='DEFAULT')

# Layers for Neural Style Transfer
style_layers, content_layers = [0, 5, 10, 19, 28], [25]
content_weight, style_weight, tv_weight = 1, 1e3, 10

# Normalization constants moved to the same device
rgb_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
rgb_std = torch.tensor([0.229, 0.224, 0.225]).to(device)

# Extract features and move the model to the target device
net = nn.Sequential(*[pretrained_net.features[i] for i in
                      range(max(content_layers + style_layers) + 1)])

# CRITICAL FIX: Move the model to GPU (cuda) to match input type
net = net.to(device)

# Set model to evaluation mode
net.eval()

output_area = widgets.Output()

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 219MB/s]


In [4]:
def preprocess(img, image_shape):

    img = img.convert("RGB")
    transforms = torchvision.transforms.Compose([
        torchvision.transforms.Resize(image_shape),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=rgb_mean, std=rgb_std)])
    return transforms(img).unsqueeze(0)

def postprocess(img):
    img = img[0].to(rgb_std.device)
    img = torch.clamp(img.permute(1, 2, 0) * rgb_std + rgb_mean, 0, 1)
    return torchvision.transforms.ToPILImage()(img.permute(2, 0, 1))

def extract_features(X, content_layers, style_layers):
    contents = []
    styles = []
    for i in range(len(net)):
        X = net[i](X)
        if i in style_layers:
            styles.append(X)
        if i in content_layers:
            contents.append(X)
    return contents, styles

def get_contents(image_shape, device):
    content_X = preprocess(content_img, image_shape).to(device)
    contents_Y, _ = extract_features(content_X, content_layers, style_layers)
    return content_X, contents_Y

def get_styles(image_shape, device):
    style_X = preprocess(style_img, image_shape).to(device)
    _, styles_Y = extract_features(style_X, content_layers, style_layers)
    return style_X, styles_Y

def content_loss(Y_hat, Y):
    return torch.square(Y_hat - Y.detach()).mean()

def gram(X):
    num_channels, n = X.shape[1], X.numel() // X.shape[1]
    X = X.reshape((num_channels, n))
    return torch.matmul(X, X.T) / (num_channels * n)

def style_loss(Y_hat, gram_Y):
    return torch.square(gram(Y_hat) - gram_Y.detach()).mean()

def tv_loss(Y_hat):
    return 0.5 * (torch.abs(Y_hat[:, :, 1:, :] - Y_hat[:, :, :-1, :]).mean() +
                  torch.abs(Y_hat[:, :, :, 1:] - Y_hat[:, :, :, :-1]).mean())

def compute_loss(X, contents_Y_hat, styles_Y_hat, contents_Y, styles_Y_gram):
    contents_l = [content_loss(Y_hat, Y) * content_weight for Y_hat, Y in zip(
        contents_Y_hat, contents_Y)]
    styles_l = [style_loss(Y_hat, Y) * style_weight for Y_hat, Y in zip(
        styles_Y_hat, styles_Y_gram)]
    tv_l = tv_loss(X) * tv_weight
    l = sum(10 * styles_l + contents_l + [tv_l])
    return contents_l, styles_l, tv_l, l

class SynthesizedImage(nn.Module):
    def __init__(self, img_shape, **kwargs):
        super(SynthesizedImage, self).__init__(**kwargs)
        self.weight = nn.Parameter(torch.rand(*img_shape))

    def forward(self):
        return self.weight


def get_inits(X, device, lr, styles_Y):
    gen_img = SynthesizedImage(X.shape).to(device)
    gen_img.weight.data.copy_(X.data)
    trainer = torch.optim.Adam(gen_img.parameters(), lr=lr)
    styles_Y_gram = [gram(Y) for Y in styles_Y]
    return gen_img(), styles_Y_gram, trainer

In [5]:
# Training function
def train(X, contents_Y, styles_Y, device, lr, num_epochs, lr_decay_epoch):
    X, styles_Y_gram, trainer = get_inits(X, device, lr, styles_Y)
    scheduler = torch.optim.lr_scheduler.StepLR(trainer, lr_decay_epoch, 0.8)

    loss_history = []
    
    # Use tqdm to show a progress bar instead of printing every epoch
    progress_bar = tqdm(range(num_epochs), desc="🎨 Styling in progress")
    
    for epoch in progress_bar:
        trainer.zero_grad()
        contents_Y_hat, styles_Y_hat = extract_features(X, content_layers, style_layers)
        contents_l, styles_l, tv_l, l = compute_loss(X, contents_Y_hat, styles_Y_hat, contents_Y, styles_Y_gram)
        l.backward()
        trainer.step()
        scheduler.step()

        # Update progress bar with the latest loss
        if (epoch + 1) % 10 == 0:
            progress_bar.set_postfix({"Loss": f"{l.item():.4f}"})
        
        loss_history.append((sum(contents_l).item(), sum(styles_l).item(), tv_l.item()))

    # Display the final image only after training is complete
    plt.figure(figsize=(10, 8))
    plt.imshow(postprocess(X))
    plt.title('Final Stylized Image')
    plt.axis('off')
    plt.show()

    return X, loss_history

def create_download_link(pil_img, filename="stylized_image.png"):
    """Generates a clickable HTML link to download the PIL image."""
    buffered = io.BytesIO()
    pil_img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    href = f'<a href="data:file/png;base64,{img_str}" download="{filename}" style="color: #007bff; font-weight: bold; font-size: 16px;">Click here to Download Stylized Image</a>'
    return HTML(href)

def start_app():
    """Initializes/Resets the UI."""
    global content_img, style_img
    output_area.clear_output()
    content_img = None
    style_img = None
    display_ui()

def run_style_transfer(b):
    """The main execution pipeline with direct value fetching."""
    global content_img, style_img

    # 1. Fetch images directly from widgets
    if content_upload.value:
        file_info = content_upload.value[0]
        content_img = Image.open(io.BytesIO(file_info['content']))
        print("Content image uploaded successfully.")  # Debugging line

    # Handle style image
    if style_dropdown.value == "Custom (Upload)" and style_upload.value:
        file_info = style_upload.value[0]
        style_img = Image.open(io.BytesIO(file_info['content']))
        print("Custom style image uploaded successfully.")  # Debugging line
    elif style_dropdown.value != "Custom (Upload)":
        style_img = Image.open(STYLE_PATHS[style_dropdown.value])
        print(f"Style image selected: {style_dropdown.value}")  # Debugging line

    # Update weights from sliders
    c_weight = content_w_slider.value
    s_weight = style_w_slider.value
    t_weight = tv_w_slider.value
    num_epochs = epoch_slider.value

    with output_area:
        clear_output()
        # Verify both images exist before starting
        if content_img is None:
            print("❌ Error: Content image is missing. Please re-upload.")
            return
        if style_img is None:
            print("❌ Error: Style image is missing. Please select one.")
            return

        print(f"🚀 Starting... Epochs: {num_epochs}, Style Weight: {s_weight:.1e}")

        # Use user-defined dimensions for the output image
        image_shape = (height_slider.value, width_slider.value)

        # Standard NST Process
        content_X, contents_Y = get_contents(image_shape, device)
        _, styles_Y = get_styles(image_shape, device)

        # Train the model
        final_X_tensor, _ = train(content_X, contents_Y, styles_Y, device, 0.3, num_epochs, 50)
        final_pil = postprocess(final_X_tensor)

        clear_output()
        plt.figure(figsize=(10, 8))
        plt.imshow(final_pil)
        plt.axis('off')
        plt.show()

        display(create_download_link(final_pil))

        restart_btn = widgets.Button(description="Try Another Style", button_style='warning', icon='redo')
        restart_btn.on_click(lambda x: start_app())
        display(restart_btn)

def display_ui():
    """Renders the dashboard with improved layout and spacing."""
    global content_upload, style_dropdown, style_upload, epoch_slider
    global content_w_slider, style_w_slider, tv_w_slider, width_slider, height_slider

    # Widget Definitions
    content_upload = widgets.FileUpload(accept='image/*', description="Upload")
    style_dropdown = widgets.Dropdown(
        options=list(STYLE_PATHS.keys()) + ["Custom (Upload)"], 
        value="Van Gogh - Starry Night",
        description="Select Style:"
    )
    style_upload = widgets.FileUpload(accept='image/*', description="Upload", layout={'display': 'none'})

    epoch_slider = widgets.IntSlider(value=500, min=100, max=2000, step=100, description='Epochs:')
    content_w_slider = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Content Weight:')
    style_w_slider = widgets.FloatLogSlider(value=1e4, base=10, min=2, max=6, step=0.5, description='Style Weight:')
    tv_w_slider = widgets.FloatSlider(value=10.0, min=0, max=50.0, step=1.0, description='TV Weight:')

    # New sliders for output image size
    width_slider = widgets.IntSlider(value=650, min=100, max=2000, step=10, description='Width:')
    height_slider = widgets.IntSlider(value=480, min=100, max=2000, step=10, description='Height:')

    confirm_btn = widgets.Button(description="Start", button_style='success', icon='paint-brush')

    # Visibility Logic
    def on_style_change(change):
        style_upload.layout.display = 'block' if style_dropdown.value == "Custom (Upload)" else 'none'

    style_dropdown.observe(on_style_change, names='value')
    confirm_btn.on_click(run_style_transfer)

    # Layout with improved styling and spacing
    ui = widgets.VBox([
        widgets.HTML("<h1 style='font-size: 24px; text-align: center;'>🎨 Neural Style Transfer Studio</h1>"),
        widgets.HTML("<h2 style='font-size: 20px;'>Upload Content Image</h2>"),
        content_upload,
        widgets.HTML("<h2 style='font-size: 20px;'>Select Style Image</h2>"),
        style_dropdown,
        style_upload,
        widgets.HTML("<h2 style='font-size: 20px;'>Training Parameters</h2>"),
        epoch_slider,
        content_w_slider,
        style_w_slider,
        tv_w_slider,
        widgets.HTML("<h2 style='font-size: 20px;'>Output Image Size</h2>"),
        width_slider,
        height_slider,
        confirm_btn,
        output_area
    ], layout={'align_items': 'center', 'padding': '20px'})

    # Display the UI
    display(ui)


In [6]:
def create_download_link(pil_img, filename="stylized_image.png"):
    """Generates a clickable HTML link to download the PIL image."""
    buffered = io.BytesIO()
    pil_img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    href = f'<a href="data:file/png;base64,{img_str}" download="{filename}" style="color: #007bff; font-weight: bold; font-size: 16px;">Click here to Download Stylized Image</a>'
    return HTML(href)

def start_app():
    """Initializes/Resets the UI."""
    global content_img, style_img
    output_area.clear_output()
    content_img = None
    style_img = None
    display_ui()

def run_style_transfer(b):
    """The main execution pipeline with direct value fetching."""
    global content_img, style_img

    # 1. Fetch images directly from widgets
    if content_upload.value:
        file_info = content_upload.value[0]
        content_img = Image.open(io.BytesIO(file_info['content']))
        print("Content image uploaded successfully.")  # Debugging line

    # Handle style image
    if style_dropdown.value == "Custom (Upload)" and style_upload.value:
        file_info = style_upload.value[0]
        style_img = Image.open(io.BytesIO(file_info['content']))
        print("Custom style image uploaded successfully.")  # Debugging line
    elif style_dropdown.value != "Custom (Upload)":
        style_img = Image.open(STYLE_PATHS[style_dropdown.value])
        print(f"Style image selected: {style_dropdown.value}")  # Debugging line

    # Update weights from sliders
    c_weight = content_w_slider.value
    s_weight = style_w_slider.value
    t_weight = tv_w_slider.value
    num_epochs = epoch_slider.value

    with output_area:
        clear_output()
        # Verify both images exist before starting
        if content_img is None:
            print("❌ Error: Content image is missing. Please re-upload.")
            return
        if style_img is None:
            print("❌ Error: Style image is missing. Please select one.")
            return

        print(f"🚀 Starting... Epochs: {num_epochs}, Style Weight: {s_weight:.1e}")

        # Use user-defined dimensions for the output image
        image_shape = (height_slider.value, width_slider.value)

        # Standard NST Process
        content_X, contents_Y = get_contents(image_shape, device)
        _, styles_Y = get_styles(image_shape, device)

        # Train the model
        final_X_tensor, _ = train(content_X, contents_Y, styles_Y, device, 0.3, num_epochs, 50)
        final_pil = postprocess(final_X_tensor)

        clear_output()
        plt.figure(figsize=(10, 8))
        plt.imshow(final_pil)
        plt.axis('off')
        plt.show()

        display(create_download_link(final_pil))

        exit_btn = widgets.Button(description="Exit", button_style='danger', icon='times')
        exit_btn.on_click(exit_app)
        display(exit_btn)

def exit_app(b):
    """Handles exiting the app."""
    with output_area:
        clear_output()
        print("👋 The app has been closed. Thank you for using it!")

def display_ui():
    """Renders the dashboard with improved layout and spacing."""
    global content_upload, style_dropdown, style_upload, epoch_slider
    global content_w_slider, style_w_slider, tv_w_slider, width_slider, height_slider

    # Widget Definitions
    content_upload = widgets.FileUpload(accept='image/*', description="Upload")
    style_dropdown = widgets.Dropdown(
        options=list(STYLE_PATHS.keys()) + ["Custom (Upload)"], 
        value="Van Gogh - Starry Night",
        description="Select Style:"
    )
    style_upload = widgets.FileUpload(accept='image/*', description="Upload", layout={'display': 'none'})

    epoch_slider = widgets.IntSlider(value=500, min=100, max=2000, step=100, description='Epochs:')
    content_w_slider = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Content Weight:')
    style_w_slider = widgets.FloatLogSlider(value=1e4, base=10, min=2, max=6, step=0.5, description='Style Weight:')
    tv_w_slider = widgets.FloatSlider(value=10.0, min=0, max=50.0, step=1.0, description='TV Weight:')

    # New sliders for output image size
    width_slider = widgets.IntSlider(value=650, min=100, max=2000, step=10, description='Width:')
    height_slider = widgets.IntSlider(value=480, min=100, max=2000, step=10, description='Height:')

    confirm_btn = widgets.Button(description="Start", button_style='success', icon='paint-brush')

    # Visibility Logic
    def on_style_change(change):
        style_upload.layout.display = 'block' if style_dropdown.value == "Custom (Upload)" else 'none'

    style_dropdown.observe(on_style_change, names='value')
    confirm_btn.on_click(run_style_transfer)

    # Layout with improved styling and spacing
    ui = widgets.VBox([
        widgets.HTML("<h1 style='font-size: 24px; text-align: center;'>🎨 Neural Style Transfer Studio</h1>"),
        widgets.HTML("<h2 style='font-size: 20px;'>Upload Content Image</h2>"),
        content_upload,
        widgets.HTML("<h2 style='font-size: 20px;'>Select Style Image</h2>"),
        style_dropdown,
        style_upload,
        widgets.HTML("<h2 style='font-size: 20px;'>Training Parameters</h2>"),
        epoch_slider,
        content_w_slider,
        style_w_slider,
        tv_w_slider,
        widgets.HTML("<h2 style='font-size: 20px;'>Output Image Size</h2>"),
        width_slider,
        height_slider,
        confirm_btn,
        output_area
    ], layout={'align_items': 'center', 'padding': '20px'})

    # Display the UI
    display(ui)

In [7]:
# --- Initial Execution ---
start_app()